# Voice-to-LLM Pipeline

This notebook demonstrates a Speech-to-Text to LLM pipeline using:
- **Speech-to-Text**: Whisper (OpenAI or ARTPARK-IISc models via Hugging Face)
- **LLM**: Qwen2.5 or Llama (via Transformers & BitsAndBytes)

**Note**: Ensure you are connected to a GPU Runtime (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
# @title 1. Install Dependencies
!pip install -q git+https://github.com/huggingface/transformers.git accelerate bitsandbytes ffmpeg-python scipy
# Update torch components together to match versions
!pip install -q --upgrade torch torchvision torchaudio

In [ ]:
# @title 2. Imports & Configuration
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import IPython
from google.colab import output
import base64
import logging

# Suppress warnings
logging.getLogger("transformers").setLevel(logging.ERROR)

# @markdown ### Model Selection
stt_model_name = "ARTPARK-IISc/whisper-small-vaani-kannada" # @param ["openai/whisper-base", "openai/whisper-small", "openai/whisper-large-v3", "ARTPARK-IISc/whisper-small-vaani-kannada", "ARTPARK-IISc/whisper-large-v3-vaani-hindi"]
language = "kannada" # @param ["english", "kannada", "hindi", "auto"]
llm_model_id = "Qwen/Qwen2.5-3B-Instruct" # @param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "meta-llama/Llama-3.2-1B-Instruct"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# @title 3. Initialize Models

print(f"Loading STT Model: {stt_model_name}...")
# Load Speech-to-Text Model (Whisper)
transcriber = pipeline(
    "automatic-speech-recognition",
    model=stt_model_name,
    device=device,
    chunk_length_s=30,
)

print(f"Loading LLM: {llm_model_id}...")
# Load LLM with 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

try:
    tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
    model = AutoModelForCausalLM.from_pretrained(
        llm_model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
except Exception as e:
    print(f"Error loading LLM: {e}")
    print("Trying with standard loading (might OOM on T4 if model is large)...")
    tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
    model = AutoModelForCausalLM.from_pretrained(llm_model_id, device_map="auto", torch_dtype=torch.float16)

print("Models loaded successfully!")

In [ ]:
# @title 4. Audio Recording Helper

AUDIO_HTML = """
<script>
var my_div = document.createElement('DIV');
var my_p = document.createElement('P');
var my_btn = document.createElement('BUTTON');
var t = document.createTextNode('Press to Start Recording');

my_btn.appendChild(t);
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var base64data = 0;
var reader;
var recorder, gumStream;
var recordButton = my_btn;

var handleSuccess = function(stream) {
  gumStream = stream;
  var options = {
    //bitsPerSecond: 8000, //chrome appears to ignore, always 48k
    mimeType : 'audio/webm;codecs=opus'
    //mimeType : 'audio/webm;codecs=pcm'
  };
  recorder = new MediaRecorder(stream, options);
  recorder.ondataavailable = function(e) {
    var url = URL.createObjectURL(e.data);
    var preview = document.createElement('audio');
    preview.controls = true;
    preview.src = url;
    document.body.appendChild(preview);

    reader = new FileReader();
    reader.readAsDataURL(e.data);
    reader.onloadend = function() {
      base64data = reader.result;
      //console.log("Inside FileReader:" + base64data);
    }
  };
  recorder.start();
};

recordButton.innerText = "Recording... press to stop";

navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);


function toggleRecording() {
  if (recorder && recorder.state == "recording") {
      recorder.stop();
      gumStream.getAudioTracks()[0].stop();
      recordButton.innerText = "Saving the recording... pls wait!"
  }
}

// https://stackoverflow.com/a/951057
function sleep(ms) {
  return new Promise(resolve => setTimeout(resolve, ms));
}

var data = new Promise(resolve=>{
  //recordButton.onclick = ()=>
  //{
    //toggleRecording()

    recordButton.onclick = ()=>{
      toggleRecording()

      sleep(2000).then(() => {
        // wait 2000ms for the data to be available...
        // ideally this should use something like await reader.onloadend
        resolve(base64data.toString())

      });

    }
  //}
});
      
</script>
"""

def get_audio():
  display(IPython.display.HTML(AUDIO_HTML))
  data = output.eval_js("data")
  binary = base64.b64decode(data.split(',')[1])
  
  process_id = 0
  filename = 'audio_input.wav'
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
# @title 5. Run Pipeline
import scipy.io.wavfile
import numpy as np
from transformers import pipeline

# Record Audio
print("Speak now...")
try:
    audio_file = get_audio()
    print("Recording saved.")

    # 1. Transcribe
    print("Transcribing...")
    # For pipeline, we just pass the filename. 
    # If language is 'auto', we let the model detect (default behavior often works, 
    # but for specific language models like 'vaani-kannada', it's implicit).
    # If explicit language is needed for multilingual whisper (openai/whisper-large), we can pass generate_kwargs
    
    generate_kwargs = {}
    if language != "auto" and "openai" in stt_model_name:
        generate_kwargs = {"language": language}
        
    result = transcriber(audio_file, generate_kwargs=generate_kwargs)
    transcribed_text = result["text"]
    print(f"\n📝 Transcribed Text: {transcribed_text}")

    # 2. LLM Inference
    print("\n🤖 Generating Response...")
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Reply concisely."},
        {"role": "user", "content": transcribed_text}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"\n💡 Assitant Response:\n{response}")

except Exception as e:
    print(f"An error occurred: {e}")